In [2]:
import numpy as np #array operations
from scipy.stats import norm #normal distribution functions
from scipy.optimize import brentq #root finding for implied volatility

In [3]:
def d1(S,K,T,r,sigma):
    return (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
def d2(S,K,T,r,sigma):
    return d1(S,K,T,r,sigma) - sigma * np.sqrt(T)
    

In [5]:
#Test cell testing defined equations
S=182
K=185
T= 90/365 # 90 days to maturity (T measured in years)
r=.053 # risk free rate
sigma= .25 #volatility
print(f"d1= {d1(S,K,T,r,sigma):.4f}")
print(f"d2= {d2(S,K,T,r,sigma):.4f}")

d1= 0.0356
d2= -0.0885


d1 represents the number of standard deviations a stock is from the strike, adjusting for time and "drift"
d2 risk neutral probability input for the stock option expiring "in the money."
With our values, it shows there is a slightly less than 50% chance the stock will finish above 185$ at this period in time.

In [6]:
print(f"N(d1) = {norm.cdf(0.0356):.4f}")
print(f"N(d2) = {norm.cdf(-0.0885):.4f}")

N(d1) = 0.5142
N(d2) = 0.4647


N(d2) = probability option expires in the money
N(d1)= delta-adjusted probability, delta is how much the option price moves for every dollar change in the asset price
"drift" is the expected direction one would expect a stock to go, in this example, we are assuming it will grow at the risk free rate, and the noise from the volatility may alter that short term.
N(d1)= delta, the amount of shares you need to short to hedge one option contract

Call price = (shares needed to hedge × stock price)
           − (probability of finishing ITM × discounted strike)
           
or more simply, 

Call price = what you gain from owning the stock hedge
           − what you pay out at expiry if exercised


In [7]:
from scipy.stats import norm

print(f"N(d1) = {norm.cdf(d1(S,K,T,r,sigma)):.4f}  ← your delta (hedge ratio)")
print(f"N(d2) = {norm.cdf(d2(S,K,T,r,sigma)):.4f}  ← probability of finishing ITM")
print()
print(f"Interpretation:")
print(f"  {norm.cdf(d2(S,K,T,r,sigma))*100:.1f}% chance Apple finishes above ${K}")
print(f"  Need to own {norm.cdf(d1(S,K,T,r,sigma)):.2f} shares to hedge 1 option contract")

N(d1) = 0.5142  ← your delta (hedge ratio)
N(d2) = 0.4647  ← probability of finishing ITM

Interpretation:
  46.5% chance Apple finishes above $185
  Need to own 0.51 shares to hedge 1 option contract


Call and Put Prices
Black Scholes formula for european options
call: right to by stock at strike K
put: right to sell stock at strike K

In [8]:
def call_price(S,K,T,r,sigma):
    return S * norm.cdf(d1(S,K,T,r,sigma)) - K * np.exp(-r*T) * norm.cdf(d2(S,K,T,r,sigma))
def put_price(S,K,T,r,sigma):
    return K * np.exp(-r*T) * norm.cdf(-d2(S,K,T,r,sigma)) - S * norm.cdf(-d1(S,K,T,r,sigma))

C  =  S · N(d1)          −        K · e^(−rT) · N(d2)

S * N(d1)= S * hedge ratio
multiplying the value of the stock by the hedge proportion, this indicates the value of stock needed to hedge the stock option
"Cost of shares needed to back the option"

K* e^(-rT)* N(d2)
K* e^(-rT) is the strike price discounted back to todays value, as the strike price initially input was at maturity, which is multiplied by the probability that the option is in the money
"The expected cost of paying the strike price at maturity"

Price of call option is the difference betweeen these two terms

P  =  K · e^(−rT) · N(−d2)   −   S · N(−d1)
      ↑                            ↑
      Term 1                       Term 2

put option is opposite basically, where d1 and d2 are negative as you want to know probability that stock finishes below the strike, as that is what you look for in a put option

term 1 is expected value of receiving strike price at maturity

term 2 is the value of the stock that is delivered

In [11]:
call = round(call_price(S, K, T, r, sigma),2)

put  = round(put_price(S, K, T, r, sigma),2)

print(f"Call price : ${call:.2f}")
print(f"Put price  : ${put:.2f}")

Call price : $8.73
Put price  : $9.32


The Greeks
measures sensitivity the option price is to the inputs
- Delta : sensitivity to stock price
- Gamma : sensitivity of Delta to stock price  
- Theta : sensitivity to time
- Vega  : sensitivity to volatility
- Rho   : sensitivity to interest rate

In [12]:
#defining Delta 
def delta(S,K,T,r,sigma,option_type='call'):
    if option_type == 'call':
        return norm.cdf(d1(S,K,T,r,sigma))
    elif option_type == 'put':
        return norm.cdf(d1(S,K,T,r,sigma)) - 1
    else:
        raise ValueError("option_type must be 'call' or 'put'")

N(d1) is the hedge ratio also known as Delta, which represents the amount the option price moves based on $1 move in the stock. If it is a put option, we calculate this with N(d1)-1 as option will gain value as stock decreases.

In [13]:
# defining Gamma
def gamma(S,K,T,r,sigma):
    return norm.pdf(d1(S,K,T,r,sigma)) / (S * sigma * np.sqrt(T))

Mathmatically the second derivative, or the rate of change of delta, which is the rate of change in the options price with respect to the stock price. This value helps us understand at what rate does your hedging need to change, when the stock price moves.
The higher the value for gamma, the more frequently the rebalancing of your hedging needs to happen.

In [15]:
#Defining theta
def theta(S,K,T,r,sigma,option_type='call'):
    if option_type == 'call':
        return (-S * norm.pdf(d1(S,K,T,r,sigma)) * sigma / (2 * np.sqrt(T)) 
                - r * K * np.exp(-r*T) * norm.cdf(d2(S,K,T,r,sigma)))
    elif option_type == 'put':
        return (-S * norm.pdf(d1(S,K,T,r,sigma)) * sigma / (2 * np.sqrt(T)) 
                + r * K * np.exp(-r*T) * norm.cdf(-d2(S,K,T,r,sigma)))
    else:
        raise ValueError("option_type must be 'call' or 'put'")

Theta helps us understand the amount of money the options looses per **day** nearing maturity.

In [16]:
# defining Vega
def vega(S,K,T,r,sigma):
    return S * norm.pdf(d1(S,K,T,r,sigma)) * np.sqrt(T) / 100

Vega helps us understand the change in option price for every 1% change in the volatility of the stock. Vega stays positive for both options and puts due to the fact that options are more valuable the more the price of the stock fluctuates.

In [17]:
#defining Rho
def rho(S,K,T,r,sigma,option_type='call'):
    if option_type == 'call':
        return K * T * np.exp(-r*T) * norm.cdf(d2(S,K,T,r,sigma)) / 100
    elif option_type == 'put':
        return -K * T * np.exp(-r*T) * norm.cdf(-d2(S,K,T,r,sigma)) / 100
    else:
        raise ValueError("option_type must be 'call' or 'put'")

Rho helps us understand how the option price will change for every 1% change in the interest rate.

Now for our earlier example, we will calculate each of these.

In [19]:
call_delta = delta(S, K, T, r, sigma, "call")
put_delta  = delta(S, K, T, r, sigma, "put")
gam        = gamma(S, K, T, r, sigma)
call_theta = theta(S, K, T, r, sigma, "call")
call_vega  = vega(S, K, T, r, sigma)
call_rho   = rho(S, K, T, r, sigma, "call")


print()
print(f"Delta (call) : {call_delta:.4f}")
print(f"  → option moves ${call_delta:.2f} per $1 move in stock")
print(f"  → need to short {call_delta:.2f} shares to hedge 1 contract")
print()
print(f"Delta (put)  : {put_delta:.4f}")
print(f"  → option moves ${put_delta:.2f} per $1 move in stock")
print()
print(f"Gamma        : {gam:.6f}")
print(f"  → delta changes {gam:.4f} per $1 move in stock")
print()
print(f"Theta        : {call_theta:.4f} per day")
print(f"  → option loses ${abs(call_theta):.4f} per day")
print(f"  → over 30 days: ${abs(call_theta)*30:.2f} time decay")
print()
print(f"Vega         : {call_vega:.4f} per 1% vol move")
print(f"  → option gains ${call_vega:.4f} per 1% rise in vol")
print()
print(f"Rho          : {call_rho:.4f} per 1% rate move")
print(f"  → option gains ${call_rho:.4f} per 1% rise in rates")


Delta (call) : 0.5142
  → option moves $0.51 per $1 move in stock
  → need to short 0.51 shares to hedge 1 contract

Delta (put)  : -0.4858
  → option moves $-0.49 per $1 move in stock

Gamma        : 0.017646
  → delta changes 0.0176 per $1 move in stock

Theta        : -22.7635 per day
  → option loses $22.7635 per day
  → over 30 days: $682.91 time decay

Vega         : 0.3603 per 1% vol move
  → option gains $0.3603 per 1% rise in vol

Rho          : 0.2092 per 1% rate move
  → option gains $0.2092 per 1% rise in rates


***Implied Volatility***

Given the market prices for an option, what is the implied volatility?

In [22]:
def implied_volatility(market_price, S, K, T, r, option_type="call"):
    if option_type == "call":
        intrinsic = max(S - K * np.exp(-r * T), 0)
        price_fn  = call_price
    else:
        intrinsic = max(K * np.exp(-r * T) - S, 0)
        price_fn  = put_price
    
    # Can't solve if price is below intrinsic value
    if market_price <= intrinsic:
        return np.nan
    
    # Objective function — we want this to equal zero
    objective = lambda sigma: price_fn(S, K, T, r, sigma) - market_price
    
    try:
        iv = brentq(objective, 1e-6, 10.0, xtol=1e-6, maxiter=500)
        return round(iv, 6)
    except ValueError:
        return np.nan

In [23]:
# Simulate observing a market price and solving for IV
observed_price = call_price(S, K, T, r, 0.30)   # market pricing at 30% vol
your_forecast  = 0.25                             # you think vol is only 25%

iv = implied_volatility(observed_price, S, K, T, r, "call")

print(f"===== Implied Volatility Solver =====")
print()
print(f"Observed market price  : ${observed_price:.4f}")
print(f"Solved implied vol     : {iv*100:.2f}%")
print(f"Your forecast vol      : {your_forecast*100:.2f}%")
print(f"Difference             : {(your_forecast - iv)*100:+.2f}%")
print()
if your_forecast < iv:
    print(f"  ► Market OVERPRICING  → SELL the call")
    print(f"    Market implies {iv*100:.1f}% vol, you forecast {your_forecast*100:.1f}%")
elif your_forecast > iv:
    print(f"  ► Market UNDERPRICING → BUY the call")
    print(f"    Market implies {iv*100:.1f}% vol, you forecast {your_forecast*100:.1f}%")
else:
    print(f"  ► Fairly priced — no edge")

===== Implied Volatility Solver =====

Observed market price  : $10.5276
Solved implied vol     : 30.00%
Your forecast vol      : 25.00%
Difference             : -5.00%

  ► Market OVERPRICING  → SELL the call
    Market implies 30.0% vol, you forecast 25.0%


Computing Note: Volatility is found within the cumulative distributions of d1 and d2, so we cannot pull it out algebraically. using brentq closes in on the volatility values by narrowing its range until the market price matches the value for the calculated call or option price.

## Developing the Full Options Chain

In [24]:
def options_chain(S, T, r, sigma, n_strikes=11, width=0.20):
    low     = S * (1 - width)
    high    = S * (1 + width)
    strikes = np.linspace(low, high, n_strikes)
    
    chain = []
    for K in strikes:
        if abs(S - K) / S < 0.02:
            moneyness = "ATM"
        elif S > K:
            moneyness = "ITM"
        else:
            moneyness = "OTM"
        
        chain.append({
            "strike"    : round(K, 2),
            "call"      : round(call_price(S, K, T, r, sigma), 4),
            "put"       : round(put_price(S, K, T, r, sigma), 4),
            "delta"     : round(delta(S, K, T, r, sigma), 4),
            "gamma"     : round(gamma(S, K, T, r, sigma), 6),
            "vega"      : round(vega(S, K, T, r, sigma), 4),
            "moneyness" : moneyness,
        })
    
    return pd.DataFrame(chain)

In [26]:
import pandas as pd

# Market chain — what the market prices at 30% vol
mkt_chain  = options_chain(S, T, r, sigma=0.30)

# Your chain — what you think fair value is at 25% vol
your_chain = options_chain(S, T, r, sigma=0.25)

# Add your prices and edge to the market chain
mkt_chain["your_call"] = your_chain["call"]
mkt_chain["edge"]      = (mkt_chain["your_call"] - mkt_chain["call"]).round(4)
mkt_chain["signal"]    = mkt_chain["edge"].apply(
    lambda e: "BUY ▲" if e > 0.05 else ("SELL ▼" if e < -0.05 else "—")
)

print()
print(mkt_chain[["strike","moneyness","call","your_call","edge","signal","delta","vega"]].to_string(index=False))


 strike moneyness    call  your_call    edge signal  delta   vega
 145.60       ITM 38.8684    38.5104 -0.3580 SELL ▼ 0.9516 0.0909
 152.88       ITM 32.3304    31.6937 -0.6367 SELL ▼ 0.9087 0.1484
 160.16       ITM 26.2468    25.2654 -0.9814 SELL ▼ 0.8462 0.2142
 167.44       ITM 20.7590    19.4258 -1.3332 SELL ▼ 0.7648 0.2778
 174.72       ITM 15.9767    14.3582 -1.6185 SELL ▼ 0.6687 0.3278
 182.00       ATM 11.9588    10.1803 -1.7785 SELL ▼ 0.5644 0.3558
 189.28       OTM  8.7062     6.9174 -1.7888 SELL ▼ 0.4597 0.3587
 196.56       OTM  6.1677     4.5048 -1.6629 SELL ▼ 0.3615 0.3386
 203.84       OTM  4.2557     2.8141 -1.4416 SELL ▼ 0.2747 0.3014
 211.12       OTM  2.8631     1.6888 -1.1743 SELL ▼ 0.2021 0.2546
 218.40       OTM  1.8807     0.9755 -0.9052 SELL ▼ 0.1442 0.2052
